# Phase 1 — Zillow + FRED Ingest, Join, and Coverage Audit

**Purpose.** Load the two public sources into Postgres and join them in SQL:

1. **Zillow ZHVI** (county, monthly): reshape to long format, build the 5-digit county FIPS key, keep each county's own published window, and load it as `zhvi_monthly`.
2. **FRED `MORTGAGE30US`** (national, weekly): take the `.mean()` of each month's weekly readings, dated at month-end, and load it as `mortgage_monthly`.
3. **Join**: done in SQL (`sql/002_join_check.sql`), not as a pandas merge.

Two conventions are decided here, and everything downstream depends on them:

- **Every month is stored as a month-end date**, which is how Zillow publishes its columns. The join is then a plain equality. Mixing month-start and month-end silently gives zero matches in SQL.
- **Empty values inside a county's published window stay in as empty rows.** They are never dropped quietly and never turned into zeros. Later phases decide explicitly what to do with them.

**Expected after a full run** (Zillow file downloaded in October 2026, so figures drift slightly as Zillow publishes new months): `zhvi_monthly` has 709,996 rows across 3,071 counties, of which 1,043 are internal gaps with an empty ZHVI. The quarterly decline-label figures are checked in Section 8 below and again in SQL.

Expects `sql/001_schema.sql` already applied to your database. Connection details come from a `.env` file in the repo root and are never hardcoded here.

## 1. Setup & Connection

In [ ]:
import os
import pandas as pd
import requests
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

pd.set_option("display.max_columns", 20)
load_dotenv(override=True)

DB_USER = os.environ.get("PGUSER", "postgres")
DB_PASSWORD = os.environ.get("PGPASSWORD", "")
DB_HOST = os.environ.get("PGHOST", "localhost")
DB_PORT = os.environ.get("PGPORT", "5432")
# The .env value wins over any stale shell variable (override=True).
DB_NAME = os.environ.get("PGDATABASE", "housing_risk_sql")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

with engine.connect() as conn:
    print(conn.execute(text("SELECT current_database(), version();")).fetchone())

**Expected output:** the name of the database you meant and a PostgreSQL version. If the name is not the one you meant, stop here and fix `.env` before continuing: every step after this loads real data into whatever database this cell connects to.

## 2. Zillow ZHVI Ingest

In [2]:
ZILLOW_URL = (
    "https://files.zillowstatic.com/research/public_csvs/zhvi/"
    "County_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"
)

RAW_DIR = "../data/raw/zillow"
os.makedirs(RAW_DIR, exist_ok=True)
raw_path = os.path.join(RAW_DIR, "zhvi_county.csv")

if not os.path.exists(raw_path):
    resp = requests.get(ZILLOW_URL, timeout=60)
    resp.raise_for_status()
    with open(raw_path, "wb") as f:
        f.write(resp.content)
    print(f"Downloaded {len(resp.content):,} bytes to {raw_path}")
else:
    print(f"Using cached file at {raw_path}")

zhvi_wide = pd.read_csv(raw_path)
assert set(zhvi_wide["RegionType"].unique()) == {"county"}, "Not a county-only file!"
print("Raw shape:", zhvi_wide.shape)

Downloaded 13,579,570 bytes to ../data/raw/zillow\zhvi_county.csv
Raw shape: (3071, 329)


In [ ]:
id_cols = [c for c in zhvi_wide.columns if not c[:2].isdigit()]
date_cols = [c for c in zhvi_wide.columns if c not in id_cols]

zhvi_long = zhvi_wide.melt(
    id_vars=id_cols,
    value_vars=date_cols,
    var_name="month",
    value_name="zhvi",
)
# Zillow's column headers are month-end dates. Keep them exactly as published.
zhvi_long["month"] = pd.to_datetime(zhvi_long["month"])
zhvi_long["county_fips"] = (
    zhvi_long["StateCodeFIPS"].astype(str).str.zfill(2)
    + zhvi_long["MunicipalCodeFIPS"].astype(str).str.zfill(3)
)
zhvi_long = zhvi_long.rename(columns={"RegionName": "region_name", "State": "state"})[
    ["county_fips", "month", "zhvi", "region_name", "state"]
]

# Keep each county's own published window: its first to its last month with a value.
# Months outside that window are not gaps, the county was simply not covered yet.
# Empty months INSIDE the window are genuine reporting gaps. They stay as empty (NULL)
# rows instead of being dropped, so no later step can mistake a gap for "no decline".
published = zhvi_long.dropna(subset=["zhvi"])
window = (
    published.groupby("county_fips")["month"]
    .agg(first_month="min", last_month="max")
    .reset_index()
)
zhvi_clean = zhvi_long.merge(window, on="county_fips", how="inner")
zhvi_clean = zhvi_clean[
    (zhvi_clean["month"] >= zhvi_clean["first_month"])
    & (zhvi_clean["month"] <= zhvi_clean["last_month"])
]
zhvi_clean = (
    zhvi_clean[["county_fips", "month", "zhvi", "region_name", "state"]]
    .sort_values(["county_fips", "month"])
    .reset_index(drop=True)
)

# Convention check: every stored month must be a month-end date.
assert (zhvi_clean["month"] + pd.Timedelta(days=1)).dt.is_month_start.all(), "Not all months are month-end dates."

n_gap_rows = int(zhvi_clean["zhvi"].isna().sum())
print("Cleaned shape:", zhvi_clean.shape)
print("Unique counties:", zhvi_clean["county_fips"].nunique())
print(f"Internal gap rows kept as empty ZHVI: {n_gap_rows:,}")
print("Date range:", zhvi_clean["month"].min().date(), "to", zhvi_clean["month"].max().date())
zhvi_clean.head(3)

## 3. Coverage Audit

Per-county check: flags counties with under 6 months of published history (they cannot support the 6-month momentum feature yet) and counties with no published value in the training period. It also counts the internal gap rows. Diagnostic only: nothing is dropped here.

In [ ]:
published = zhvi_clean.dropna(subset=["zhvi"])
county_month_counts = (
    published.groupby("county_fips")["month"]
    .agg(first_month="min", last_month="max", n_months="count")
)
n_counties = zhvi_clean["county_fips"].nunique()
n_short_history = int((county_month_counts["n_months"] < 6).sum())

train_cutoff = pd.Timestamp("2021-09-30")
training_presence = (
    published[published["month"] <= train_cutoff]
    .groupby("county_fips")["month"]
    .count()
)
n_no_training_presence = n_counties - training_presence.shape[0]

print(f"Counties: {n_counties:,}")
print(f"Total county-month rows: {len(zhvi_clean):,}  (including {n_gap_rows:,} internal gaps with an empty ZHVI)")
print(f"Counties with < 6 months of published history: {n_short_history:,}")
print(f"Counties with no published value in the training period: {n_no_training_presence:,}")

## 4. Load Zillow to Postgres

`TRUNCATE` before the load, not `if_exists="append"` alone, so the notebook is safe to re-run top to bottom: a second run replaces the table content instead of failing on the primary key or piling up stale rows. Empty ZHVI values are loaded as NULL.

In [ ]:
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE zhvi_monthly;"))

zhvi_clean.to_sql(
    "zhvi_monthly",
    engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=5000,
)

with engine.connect() as conn:
    loaded = conn.execute(text("SELECT COUNT(*) FROM zhvi_monthly;")).scalar()
print(f"zhvi_monthly now has {loaded:,} rows")
assert loaded == len(zhvi_clean), "Row count in Postgres doesn't match the cleaned frame."

## 5. FRED `MORTGAGE30US` Ingest

In [7]:
FRED_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"

RAW_DIR = "../data/raw/fred"
os.makedirs(RAW_DIR, exist_ok=True)
raw_path = os.path.join(RAW_DIR, "mortgage30us.csv")

if not os.path.exists(raw_path):
    resp = requests.get(FRED_URL, timeout=60)
    resp.raise_for_status()
    with open(raw_path, "wb") as f:
        f.write(resp.content)
    print(f"Downloaded {len(resp.content):,} bytes to {raw_path}")
else:
    print(f"Using cached file at {raw_path}")

rate_weekly = pd.read_csv(raw_path)
rate_weekly.columns = ["date", "mortgage_rate"]
rate_weekly["date"] = pd.to_datetime(rate_weekly["date"])
rate_weekly["mortgage_rate"] = pd.to_numeric(rate_weekly["mortgage_rate"], errors="coerce")
rate_weekly = rate_weekly.dropna(subset=["mortgage_rate"])
print("Raw weekly shape:", rate_weekly.shape)

Downloaded 46,915 bytes to ../data/raw/fred\mortgage30us.csv
Raw weekly shape: (2895, 2)


**Resampling convention: `.mean()`, month-end (`"ME"`).** Each month's value is the mean of that month's weekly readings, dated at month-end like Zillow's columns. Taking the last weekly reading instead is a defensible alternative. It changes the direction of the month-over-month change in some months, so the choice is stated here so that it is explicit.

In [8]:
rate_monthly = (
    rate_weekly.set_index("date")["mortgage_rate"]
    .resample("ME")
    .agg(["mean", "count"])
    .reset_index()
    .rename(columns={"date": "month", "mean": "mortgage_rate", "count": "n_weeks"})
)
rate_monthly = rate_monthly.dropna(subset=["mortgage_rate"])
print("Monthly shape:", rate_monthly.shape)
rate_monthly.tail(3)

Monthly shape: (666, 3)


,month,mortgage_rate,n_weeks
663,2026-07-31,6.542000,5
664,2026-08-31,6.667500,4
665,2026-09-30,6.806667,3


## 6. Load FRED to Postgres

In [9]:
with engine.begin() as conn:
    conn.execute(text("DELETE FROM mortgage_monthly;"))

rate_monthly[["month", "mortgage_rate"]].to_sql(
    "mortgage_monthly",
    engine,
    if_exists="append",
    index=False,
)

with engine.connect() as conn:
    loaded = conn.execute(text("SELECT COUNT(*) FROM mortgage_monthly;")).scalar()
print(f"mortgage_monthly now has {loaded:,} rows")
assert loaded == len(rate_monthly), "Row count in Postgres doesn't match the cleaned frame."

mortgage_monthly now has 666 rows


## 7. Join — in SQL, not pandas

The join runs on the Postgres server in `sql/002_join_check.sql`: a plain equality join on the month-end date, with Zillow as the spine (`LEFT JOIN`). A month missing from FRED would therefore show up as an empty `mortgage_rate`, never as a silently dropped county-month. The script drops and rebuilds the table itself, so it is safe to re-run.

In [ ]:
# Run the whole SQL file through the raw database cursor, with no parameters.
raw = engine.raw_connection()
try:
    cur = raw.cursor()
    cur.execute(open("../sql/002_join_check.sql").read())
    raw.commit()
except Exception:
    raw.rollback()
    raise
finally:
    raw.close()

joined = pd.read_sql("SELECT * FROM county_month_joined;", engine)
print("Joined shape:", joined.shape)
print("Rows with missing mortgage_rate:", int(joined["mortgage_rate"].isna().sum()))

counties_before = set(zhvi_clean["county_fips"].unique())
counties_after = set(joined["county_fips"].unique())
dropped = len(counties_before - counties_after)
print("Counties dropped during join:", dropped)
assert dropped == 0, "A county was dropped during the join."
assert len(joined) == len(zhvi_clean), "Row count changed across the join."
joined.head(3)

## 8. Provenance Check

Computes the quarterly decline rates for Train, Embargo and Test straight from the joined table, and confirms that the SQL join did not corrupt anything.

**Label rule.** A county-quarter is represented by its quarter-end month (March, June, September, December), and a decline label is built only between two consecutive quarter-end months. A quarter whose end month is empty gets no label, and neither does the quarter after it, because its change would span six months. So both quarter-end months must have a value, and every label is a true quarter-to-quarter change.

In [ ]:
joined["month"] = pd.to_datetime(joined["month"])
joined["quarter"] = joined["month"].dt.to_period("Q")

# Quarter-end months only. If a quarter's end month has no ZHVI value, its own label and
# the next quarter's label both come out empty and are dropped.
zillow_q = (
    joined[joined["month"].dt.month.isin([3, 6, 9, 12])]
    [["county_fips", "quarter", "zhvi"]]
    .sort_values(["county_fips", "quarter"])
    .reset_index(drop=True)
)
# fill_method=None is written out so the labels do not depend on the pandas version.
zillow_q["pct_change"] = zillow_q.groupby("county_fips")["zhvi"].pct_change(fill_method=None)
zillow_q["decline_label"] = (zillow_q["pct_change"] < -0.02).astype(int)
zillow_q_valid = zillow_q.dropna(subset=["pct_change"]).copy()

def window_stats(mask, label):
    sub = zillow_q_valid[mask]
    rate = sub["decline_label"].mean() * 100
    print(f"{label:45s} rows={len(sub):>7,}   positive_rate={rate:.2f}%")

window_stats(pd.Series(True, index=zillow_q_valid.index), "Full history")
window_stats(zillow_q_valid["quarter"] <= pd.Period("2021Q3"), "Train (through 2021-09-30)")
window_stats(zillow_q_valid["quarter"] == pd.Period("2021Q4"), "Embargoed (2021-10-01 to 2021-12-31)")
window_stats(
    (zillow_q_valid["quarter"] >= pd.Period("2022Q1")) & (zillow_q_valid["quarter"] <= pd.Period("2026Q1")),
    "Test (2022-01-01 to 2026-03-31)",
)

**Expected with this label rule:** Train 173,277 rows (4.88%), Embargo 3,020 (0.96%), Test 52,085 (8.15%), Full history 231,453 (5.55%). The SQL check in `sql/006_checks.sql` (check 6a) computes the same figures on the server and must match them.

**Full history** is Train, Embargo and Test (228,382 rows) plus 3,071 rows for 2026-Q2. The unfinished 2026-Q3 gives no rows, because its quarter-end month (September) is not published yet. The figures drift slightly as Zillow publishes new months.

**Why not "the last month with a value" per quarter?** That shortcut lets a label span two, or four to five, months when a quarter's end month is empty. An independent recomputation of the modeling table found 448 rows built that way. They were about 25% declines against 4.9% overall, so they would have distorted the label. Requiring both quarter-end months avoids it.